# 01 Wrangle

Loads the raw UniProt Excel file, adds all derived variables, and saves a single clean dataset.

**Output:** `../output/full_dataset.csv` (also copied to `../dashboard/full_dataset.csv`)

| Column added | Source |
|---|---|
| `pLLPS_Class` | thresholds: High ≥ 0.7, Low < 0.4 |
| `Functional Group` | first YAML-matched category, else 'Other' |
| `All_Functional_Groups` | all YAML-matched categories (list) |
| `Location Categories` | parsed UniProt location string (list) |
| `Is_Membrane` | membrane-protein flag |
| `TMD_count` | transmembrane domain count (from cache) |

In [ ]:
from pathlib import Path

import pandas as pd

from llps.data import add_tmd_count, fetch_uniprot_tm_annotations, load_llps_data
from llps.functional import is_membrane_protein, parse_function_categories
from llps.location import parse_location

RAW     = Path("../Human Phase separation data.xlsx")
CACHE   = Path("../data/uniprot_tm_cache.csv")
OUT     = Path("../output/full_dataset.csv")
APP_OUT = Path("../dashboard/full_dataset.csv")

In [ ]:
df = load_llps_data(RAW)
print(f"{len(df):,} proteins  |  p(LLPS) {df['p(LLPS)'].min():.2f}–{df['p(LLPS)'].max():.2f}")
df.head(2)

In [ ]:
# pLLPS classification
df["pLLPS_Class"] = pd.cut(
    df["p(LLPS)"],
    bins=[-float("inf"), 0.4, 0.7, float("inf")],
    labels=["Low", "Medium", "High"],
).astype(str)

df["pLLPS_Class"].value_counts()

In [ ]:
# Functional categories (YAML-driven via llps/functional.py)
df["All_Functional_Groups"] = df.apply(
    lambda r: parse_function_categories(
        str(r.get("Function [CC]") or ""),
        str(r.get("Protein names") or ""),
    ),
    axis=1,
)
df["Functional Group"] = df["All_Functional_Groups"].apply(
    lambda x: x[0] if x else "Other"
)

df["Functional Group"].value_counts().head(10)

In [ ]:
# Location categories
df["Location Categories"] = df["Subcellular location [CC]"].apply(parse_location)

In [ ]:
# Membrane flag
df["Is_Membrane"] = df.apply(
    lambda r: is_membrane_protein(
        str(r.get("Function [CC]") or ""),
        str(r.get("Protein names") or ""),
        str(r.get("Subcellular location [CC]") or ""),
    ),
    axis=1,
)
print(f"Membrane: {df['Is_Membrane'].sum():,}  |  Non-membrane: {(~df['Is_Membrane']).sum():,}")

In [ ]:
# TMD count (reads from local cache — run scripts/analysis/enrich_dataset_with_tmd.py to populate)
if CACHE.exists():
    tm = fetch_uniprot_tm_annotations(df["Entry"].tolist(), cache_path=str(CACHE))
    df = add_tmd_count(df, tm)
else:
    df["TMD_count"] = 0
    print("⚠ TM cache not found — TMD_count set to 0")

df["TMD_count"].value_counts().sort_index().head(10)

In [ ]:
# Save
OUT.parent.mkdir(exist_ok=True)
df.to_csv(OUT, index=False)
APP_OUT.parent.mkdir(exist_ok=True)
df.to_csv(APP_OUT, index=False)

print(f"Saved {len(df):,} proteins  |  {df.shape[1]} columns")
print(f"  → {OUT}")
print(f"  → {APP_OUT}  (dashboard)")
print()
print(df.dtypes)